In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Imports And Style Configuration

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import umap

# Metrics
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

# Settings
import warnings
warnings.filterwarnings('ignore')

# --- Dark theme palatte
PALETTE  = ['#C0392B', '#2980B9','#27AE60','#8E44AD','#E67E22','#1ABC9C']
BG_COLOR = '#0F1117'
PANEL_COLOR = '#1A1D27'
TEXT_COLOR = '#E8E8E8'
GRID_COLOR = '#2A2D3A'
ACCENT_COLOR = '#C0392B'

plt.rcParams.update({
    'figure.facecolor':  BG_COLOR,
    'axes.facecolor':    PANEL_COLOR,
    'axes.edgecolor':    GRID_COLOR,
    'axes.labelcolor':   TEXT_COLOR,
    'xtick.color':       TEXT_COLOR,
    'ytick.color':       TEXT_COLOR,
    'text.color':        TEXT_COLOR,
    'grid.color':        GRID_COLOR,
    'grid.linewidth':    0.5,
    'grid.alpha':        0.4,
    'legend.facecolor':  PANEL_COLOR,
    'legend.edgecolor':  GRID_COLOR,
    'axes.spines.top':   False,
    'axes.spines.right': False,

})
print('Import Done Succesfully')

# 2. Load And Explore Data

# Load Training Data
train = pd.read_csv('/kaggle/input/wine-ensgti-2026/train.csv')

# Load Testing Data
test = pd.read_csv('/kaggle/input/wine-ensgti-2026/test.csv')

# Explore training data
print(f'training data shape {train.shape}')
print("\nFirst few row")
train.head()

# Dataset Information
print("\nColumn info")
print(train.info())


# Check missing values
print("\nMissing Values")
print(train.isnull().sum())

# 3. Exploratiory Data Analysis

# Split dataset
X_raw=train.drop(['id','quality'],axis=1)

# Group quality into 3 bins: 3-4 (Low), 5-6 (Medium), 7-9 (High)
train['quality_binned'] = train['quality'].apply(lambda x: 0 if x <= 4 else (1 if x <= 6 else 2))



# Descriptive Statistic
print("Descriptive Statistic")
train.describe()

# Visualization
fig = plt.figure(figsize=(16, 18), facecolor=BG_COLOR)
gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1.2, 1]) 

ax1 = fig.add_subplot(gs[0, :]) 
corr = train.corr()
mask_upper = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(corr, mask=mask_upper, ax=ax1, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=1, linecolor=BG_COLOR,
            annot=True, fmt='.2f', annot_kws={'size': 10})
ax1.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', color=TEXT_COLOR, pad=15)

# Violin Distribution
ax2 = fig.add_subplot(gs[1, 0])
train_norm = train.copy()
for col in train_norm.columns:
    mn,mx= train_norm[col].min(),train_norm[col].max()
    if mx > mn:
        train_norm[col] = (train_norm[col]-mn) / (mx-mn)
        
train_melt= train_norm.melt(var_name='Feature',value_name='Scale Value')
sns.violinplot(data=train_melt,x='Feature',y='Scale Value',
              palette=PALETTE[:len(train.columns)],
              ax=ax2,inner='quartile',linewidth=0.8)
ax2.set_title('Feature Distribution (Mix-Max Scaled)',fontsize=16, fontweight='bold', color=TEXT_COLOR)
ax2.tick_params(axis='x',rotation=50,labelsize=8)

# PCA explained Variance
ax3 = fig.add_subplot(gs[1, 1])
pca_full= Pipeline([('s',StandardScaler()),('p',PCA(random_state=42))])
pca_full.fit(X_raw)
ev= pca_full.named_steps['p'].explained_variance_ratio_
cumev = np.cumsum(ev)
comps= range(1,len(ev)+1)
ax3.bar(comps,ev*100,color=ACCENT_COLOR,alpha=0.75,zorder=3)
ax_3=ax3.twinx()
ax_3.plot(comps, cumev*100, 'o-', color='#F39C12', lw=2, ms=5)
ax_3.axhline(85, color='#1ABC9C', ls='--', lw=1.2, alpha=0.7)
ax_3.text(len(ev)-0.3, 86.5, '85%', color='#1ABC9C', fontsize=8, ha='right')
ax_3.set_ylabel('Cumulative Variance (%)', color='#F39C12', fontsize=9)
ax_3.tick_params(colors='#F39C12')
ax_3.set_facecolor(PANEL_COLOR)
ax3.set_xlabel('Principal Component'); ax3.set_ylabel('Explained Variance (%)')
ax3.set_title('PCA Explained Variance', fontsize=12, fontweight='bold')
ax3.set_xticks(list(comps))

plt.tight_layout()
plt.show()

# 4. Pipeline

# kmean with standard scaler inside pipeline
def build_kmeans_pipeline(n_clusters=3) -> Pipeline:
    return Pipeline([
        ('scaler',StandardScaler()),
        ('kmeans',KMeans(n_clusters=n_clusters,random_state=42,n_init=20)),
    ])

# dbscan with standard scaler inside pipeline
def build_dbscan_pipeline(eps=0.8,min_samples=10) -> Pipeline:
    return Pipeline([
        ('scaler',StandardScaler()),
        ('dbscan',DBSCAN(eps=eps,min_samples=min_samples)),
    ])

#  hdbscan with standard scaler inside pipeline
def build_hdbscan_pipeline(min_cluster_size=20) -> Pipeline:
    return Pipeline([
        ('scaler',StandardScaler()),
        ('hdbscan',HDBSCAN(min_cluster_size=min_cluster_size,store_centers='centroid')),
     ])

# pca with standard scaler inside pipeline
def build_pca_pipeline(n_components=2) -> Pipeline:
    return Pipeline([
        ('scaler',StandardScaler()),
        ('pca',PCA(n_components=n_components,random_state=42)),
    ])

print('Pipeline Defined')
print('  -> build_kmeans_pipeline(n_clusters)')
print('  -> build_dbscan_pipeline(eps, min_samples)')
print('  -> build_hdbscan_pipeline(min_cluster_size)')
print('  -> build_pca_pipeline(n_components)')


# 5. KMeans- Elbow Method & Silhouette Analysis

k_range = range(2,11)
inertias,silhouettes=[],[]

for k in k_range:
    pipe=build_kmeans_pipeline(k)
    pipe.fit(X_raw)
    labels = pipe.named_steps['kmeans'].labels_
    X_scaled = pipe.named_steps['scaler'].transform(X_raw)
    inertias.append(pipe.named_steps['kmeans'].inertia_)
    silhouettes.append(silhouette_score(X_scaled,labels))

k_vals=list(k_range)
best_k= k_vals[silhouettes.index(max(silhouettes))]
print(f'Best k by silhouette: {best_k} (score={max(silhouettes):.4f})')

fig,axes = plt.subplots(1,2,figsize=(16,5),facecolor=BG_COLOR)
fig.suptitle('KMeans - Optimal Cluster Selection',fontsize=14,fontweight='bold',color=TEXT_COLOR)

# Elbow
axes[0].plot(k_vals,inertias,'o-',color=ACCENT_COLOR,lw=2.5,ms=8,zorder=3)
axes[0].fill_between(k_vals,inertias,alpha=0.15,color=ACCENT_COLOR)
axes[0].axvline(3,ls='--',color='#F38C12',lw=1.5,label='Opitmal k=3')
axes[0].set_xlabel('Number of Cluster (k)',fontsize=11)
axes[0].set_ylabel('Inertia (Within-Cluster SSQ)',fontsize=11)
axes[0].set_title('Elbow Method)',fontsize=12,fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)


# Silhouette 
colors_s=[ACCENT_COLOR if s == max(silhouettes) else '#2980B9' for s in silhouettes]
axes[1].bar(k_vals,silhouettes, color=colors_s,zorder=3,edgecolor=BG_COLOR)
axes[1].plot(k_vals,silhouettes,'o-',color='#F39C12',lw=1.5,ms=6,zorder=4)
axes[1].set_xlabel('Number of Clusters (k)',fontsize=11)
axes[1].set_ylabel('Silhouette Score',fontsize=11)
axes[1].set_title('Silhouette Score per k',fontsize=11,fontweight='bold')
axes[1].grid(axis='y',alpha=0.3)

plt.tight_layout(); plt.show()



# We use 2 * number of features for min_samples (2 * 11 = 22)
k = 22
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

neigh = NearestNeighbors(n_neighbors=k)
nbrs = neigh.fit(X_scaled)
distances, indices = nbrs.kneighbors(X_scaled)

# Plot the distances to find the "elbow"
distances = np.sort(distances[:, k-1], axis=0)
plt.plot(distances)
plt.axhline(y=2.0, color='r', linestyle='--') # Example threshold
plt.title('K-Distance Plot (Look for the "Knee")')
plt.show()

# 6. Fit All Clustering Models

# KMeans
km_pipe = build_kmeans_pipeline(n_clusters=3)
km_pipe.fit(X_raw)
kmeans_labels = km_pipe.named_steps['kmeans'].labels_
print(f"KMeans -> Clusters: {len(set(kmeans_labels))} Distribution: {dict(zip(*np.unique(kmeans_labels,return_counts=True)))}")


# DBSCAN
db_pipe= build_dbscan_pipeline(eps=2.25,min_samples=22)
db_pipe.fit(X_raw)
dbscan_labels= db_pipe.named_steps['dbscan'].labels_
n_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
print(f"DBSCAN -> Clusters: {n_db} Noise: {(dbscan_labels==-1).sum()}")

# HDBSCAN

hdb_pipe = build_hdbscan_pipeline(min_cluster_size=20)
hdb_pipe.fit(X_raw)
hdbscan_labels = hdb_pipe.named_steps['hdbscan'].labels_
n_hdb = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
print(f"HDBSCAN -> Clusters: {n_hdb} Noise: {(hdbscan_labels==-1).sum()}")


# 7. Dimensionality Reduction (PCA + t-SNE + umap)

# PCA 2D
pca_pipe = build_pca_pipeline(n_components=2)
pca_2d = pca_pipe.fit_transform(X_raw)
print(f"PCA 2D shape: {pca_2d.shape}")

# t-SNE 2D 
X_scaled_tsne = StandardScaler().fit_transform(X_raw)
tsne = TSNE(n_components=2,perplexity=40,random_state=42,max_iter=1000)
tsne_2d = tsne.fit_transform(X_scaled_tsne)
print(f't-SNE 2D shape: {tsne_2d.shape}')

# umap 2D
reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, n_components=2, random_state=42)
umap_2d = reducer.fit_transform(X_scaled)
print(f't-SNE 2D shape: {umap_2d.shape}')

# 8. PCA Scatter - Cluster Comparison

fig, axes = plt.subplots(2,2,figsize=(14,10), facecolor=BG_COLOR)
fig.suptitle('PCA Projection - Cluster Comparison',fontsize=15,
            fontweight='bold',color=TEXT_COLOR)
configs= [
    ('KMeans (k=3)',   kmeans_labels),
    ('DBSCAN',         dbscan_labels),
    ('HDBSCAN',        hdbscan_labels),
    ('Ground-Truth Cluster', train['quality_binned'].values ),
]

for ax, (title,labels) in zip(axes.flatten(),configs):
    unique_labels = sorted(set(labels))
    for i , lbl in enumerate(unique_labels):
        mask = labels == lbl
        clr = '#444444' if lbl == -1 else PALETTE[i % len(PALETTE)]
        alpha = 0.30 if lbl == -1 else 0.75
        size = 12 if lbl == -1 else 22
        mkr = 'x' if lbl == -1 else 'o'
        lstr = 'Noise' if lbl == -1 else f'Cluster {lbl}'
        ax.scatter(pca_2d[mask,0],pca_2d[mask,1], c=clr,s=size,
                  alpha=alpha, marker=mkr,label=lstr,linewidths=0.5)
    ax.set_title(title,fontsize=11,fontweight='bold')
    ax.set_xlabel('PC1',fontsize=9); ax.set_ylabel('PC2',fontsize=9)
    ax.legend(fontsize=8,framealpha=0.6); ax.grid(alpha=0.25)

plt.tight_layout(); plt.show()

# 9. UMAP Scatter - Cluster Comparison

fig, axes = plt.subplots(2,2,figsize=(14,10), facecolor=BG_COLOR)
fig.suptitle('UMAP Projection - Cluster Comparison',fontsize=15,
            fontweight='bold',color=TEXT_COLOR)
configs= [
    ('KMeans (k=3)',   kmeans_labels),
    ('DBSCAN',         dbscan_labels),
    ('HDBSCAN',        hdbscan_labels),
    ('Ground-Truth Cluster', train['quality_binned'].values ),
]

for ax, (title,labels) in zip(axes.flatten(),configs):
    unique_labels = sorted(set(labels))
    for i , lbl in enumerate(unique_labels):
        mask = labels == lbl
        clr = '#444444' if lbl == -1 else PALETTE[i % len(PALETTE)]
        alpha = 0.30 if lbl == -1 else 0.75
        size = 12 if lbl == -1 else 22
        mkr = 'x' if lbl == -1 else 'o'
        lstr = 'Noise' if lbl == -1 else f'Cluster {lbl}'
        ax.scatter(umap_2d[mask,0],umap_2d[mask,1], c=clr,s=size,
                  alpha=alpha, marker=mkr,label=lstr,linewidths=0.5)
    ax.set_title(title,fontsize=11,fontweight='bold')
    ax.set_xlabel('PC1',fontsize=9); ax.set_ylabel('PC2',fontsize=9)
    ax.legend(fontsize=8,framealpha=0.6); ax.grid(alpha=0.25)

plt.tight_layout(); plt.show()

# 10. t-SNE Scatter - Non-Linear Projection

fig, axes= plt.subplots(1,3, figsize=(20,6),facecolor=BG_COLOR)
fig.suptitle('t-SNE Projection - Cluster Comparison',fontsize=15,
            fontweight='bold',color=TEXT_COLOR)

for ax, (title,labels) in zip(axes,[
    ('KMeans', kmeans_labels),
    ('DBSCAN', dbscan_labels),
    ('HDBSCAN',hdbscan_labels),
]):
    for i , lbl in enumerate(sorted(set(labels))):
        mask = labels== lbl
        clr = '#444444' if lbl == -1 else PALETTE[i % len(PALETTE)]
        lstr = 'Noise' if lbl == -1 else f'Cluster {lbl}'
        ax.scatter(tsne_2d[mask,0],tsne_2d[mask,1],c=clr,s=18,
                  alpha=0.3   if lbl == -1 else 0.75, label=lstr,linewidths=0)
    ax.set_title(f't-SNE - {title}', fontsize=11,fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=8 , framealpha=0.6); ax.grid(alpha=0.25)

plt.tight_layout(); plt.show()

# 1. Create a DataFrame with labels and original features
cluster_analysis = X_raw.copy()
cluster_analysis['Cluster'] = kmeans_labels

# 2. Calculate the average of each feature per cluster
cluster_means = cluster_analysis.groupby('Cluster').mean()

# 3. Normalize the data (Z-score) so we can compare different units (like pH vs Alcohol)
# This shows how many standard deviations a cluster is from the dataset average
cluster_means_scaled = (cluster_means - X_raw.mean()) / X_raw.std()

# 4. Plot the "Cluster Personality" Heatmap
plt.figure(figsize=(12, 6), facecolor=BG_COLOR)
sns.heatmap(cluster_means_scaled, annot=True, cmap='RdYlGn', center=0, fmt='.2f')
plt.title('KMeans Cluster Profiles (The "Personality" of each group)', fontsize=15, fontweight='bold')
plt.ylabel('KMeans Cluster')
plt.show()

# 5. Check Alignment with Binned Quality
comparison = pd.crosstab(train['quality_binned'], kmeans_labels, normalize='columns') * 100
plt.figure(figsize=(8, 5))
sns.heatmap(comparison, annot=True, fmt='.1f', cmap='Purples')
plt.title('Percentage of Quality Bins in each Cluster', fontsize=12)
plt.ylabel('Actual Quality (0:Low, 1:Med, 2:High)')
plt.xlabel('KMeans Cluster')
plt.show()